LoRA实现

output = (W+ΔW)*x+b
       = W*x + ΔW*x + b
       = W*x + α/r * (AB)*x + b

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [10]:
class LinearLoRALayer(nn.Module):
    def __init__(self,
        in_features,
        out_features,
        merge = False,
        rank = 8,
        lora_alpha = 16,
        dropout = 0.1,
    ):
        super().__init__()

        self.in_features = in_features
        self.out_features = out_features
        self.merge = merge
        self.rank = rank

        # W的shape (out_featrues,in_features)
        # Linear内部的乘法是：x@W^T + b
        self.Linear = nn.Linear(in_features,out_features)

        if rank > 0:
            self.lora_a = nn.Parameter(
                torch.zeros(out_features,rank)
            )
            # @春风归无期 提醒我 @用代码打点酱油的chaofa : 在调用凯明初始化的时候注释里写的高斯分布，调用的却是均匀分布，而且参数a的值设置的是根号5，但a表示的是leaky relu的负斜率系数，一般是0.01这样的小值，不可能超过1
            nn.init.kaiming_normal_(self.lora_a, a=0.01)

            self.lora_b = nn.Parameter(
                torch.zeros(rank,in_features)
            )
            self.scale = lora_alpha/rank #缩放参数

            # linear 需要设置为不可以训练
            # 只训练增量，不训练原本的模型
            self.Linear.weight.requires_grad = False
            self.Linear.bias.requires_grad = False

        # dropout = 0 - x = x
        # dropout ≠ 0 - x = self.dropout(x)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

        if merge:
            self.merge_weight()
    
    def merge_weight(self,):
        if self.merge and self.rank > 0:
            self.Linear.weight.data += self.scale*(self.lora_a @ self.lora_b)
    
    def unmerge_weight(self, ):
        if self.rank > 0:
            self.Linear.weight.data -= self.scale * (self.lora_a @ self.lora_b)

    def forward(self,x):
        # 不合并 - 训练阶段
        if self.rank > 0 and not self.merge:
            output = self.Linear(x) + self. scale*(x @ (self.lora_a @ self.lora_b).T)
        elif self.rank > 0 and self.merge:
            output = self.Linear(x)
        else:
            output = self.Linear(x)

        return self.dropout(output)
    

In [18]:
# 写一段测试代码
# Test the LoRALinear layer
batch_size = 32
seq_len = 128
in_features = 768
out_features = 512
rank = 8
lora_alpha = 16
dropout = 0.0

# Create a test input
x = torch.randn(batch_size, seq_len, in_features)

# Test regular mode (no merge)
lora_layer = LinearLoRALayer(
    in_features=in_features,
    out_features=out_features,
    rank=rank,
    lora_alpha=lora_alpha,
    dropout=dropout,
    merge=False
)

# Forward pass
output = lora_layer(x)
print(f"Output shape (no merge): {output.shape}")  # Should be [batch_size, seq_len, out_features]

# Test merged mode
lora_layer_merged = LinearLoRALayer(
    in_features=in_features,
    out_features=out_features,
    rank=rank,
    lora_alpha=lora_alpha,
    dropout=dropout,
    merge=True
)

# Forward pass with merged weights
output_merged = lora_layer_merged(x)
print(f"Output shape (merged): {output_merged.shape}")  # Should be [batch_size, seq_len, out_features]

# Test weight merging/unmerging
lora_layer.merge_weight()
output_after_merge = lora_layer(x)
lora_layer.unmerge_weight()
output_after_unmerge = lora_layer(x)

print("Max difference after merge/unmerge cycle:", 
      torch.max(torch.abs(output - output_after_unmerge)).item())

Output shape (no merge): torch.Size([32, 128, 512])
Output shape (merged): torch.Size([32, 128, 512])
Max difference after merge/unmerge cycle: 0.0


In [ ]:
import copy
import torch

batch_size = 32
seq_len = 128
in_features = 768
out_features = 512
rank = 8
lora_alpha = 16
dropout = 0.1   # 先关掉dropout

x = torch.randn(batch_size, seq_len, in_features)

# 建一个基础层
lora_layer = LinearLoRALayer(
    in_features=in_features,
    out_features=out_features,
    rank=rank,
    lora_alpha=lora_alpha,
    dropout=dropout,
    merge=False
)

'''
    有些层在：

    train() 模式
    eval() 模式

    下，行为是不一样的，最典型就是：

    Dropout
    训练时会随机丢弃一部分神经元
    推理时不会丢，直接原样通过

    BatchNorm
    训练时用当前 batch 的均值方差
    推理时用训练过程中累计的均值方差
''' 

lora_layer.eval() # 避免随机性-Dropout的干扰

# 拷贝出一份完全相同权重的层
lora_layer_merged = copy.deepcopy(lora_layer)
lora_layer_merged.merge = True
lora_layer_merged.merge_weight()
lora_layer_merged.eval()

with torch.no_grad(): # 不要记录梯度和计算图，节省开销
    output_no_merge = lora_layer(x)
    output_merge = lora_layer_merged(x)

print("Max difference:", torch.max(torch.abs(output_no_merge - output_merge)).item())

Max difference: 0.0
